In [ ]:
import pandas as pd
import requests
import time
import re
from datetime import datetime

# Caminho do arquivo .xls
arquivo = r"D:\Cotic\CONFERE_CUSTODIADOS_26_02_2026 16_18_19_UPF.xls"

# Lê a planilha
df = pd.read_excel(arquivo, sheet_name="confereCustodiado", engine="xlrd")
ids_raw = df.iloc[:, 0].astype(str).str.strip()

# Normaliza os IDs
ids = []
for val in ids_raw:
    if val and val.strip():
        try:
            ids.append(int(val))
        except ValueError:
            print(f"Valor não convertido: {val}")

print(f"Total de IDs capturados: {len(ids)}")

# Login para obter token (XML)
login_url = "http://172.18.4.46:8080/sispen_api/login/login"
login_payload = {"username": "elisangela.helcias", "password": "30069218"}
headers_login = {"Content-Type": "application/json"}

resp_login = requests.post(login_url, json=login_payload, headers=headers_login)
if resp_login.status_code == 200:
    match = re.search(r"<token>(.*?)</token>", resp_login.text)
    if match:
        token = match.group(1)
        TOKEN = f"Bearer {token}"
    else:
        raise Exception("Token não encontrado na resposta XML")
else:
    raise Exception(f"Falha no login: {resp_login.status_code}")

headers = {"Authorization": TOKEN, "Accept": "application/json"}

resultados = []

# Loop pelos IDs
for pessoa_id in ids:
    try:
        # 1. Obter id_real
        url_real = f"http://172.18.4.46:8080/sispen_api/custodiado?page=0&size=10&ativo=true&pessoaId={pessoa_id}"
        resp_real = requests.get(url_real, headers=headers)

        pessoa_nome = ""
        numero_processo = "Sem processo"
        custodiado = None

        if resp_real.status_code == 200:
            data_real = resp_real.json()
            entities_real = data_real.get("entities", [])
            if not entities_real:
                print(f"{pessoa_id} - Não encontrado no endpoint custodiado")
                continue

            id_real = entities_real[0]["id"]
            pessoa_nome = entities_real[0].get("pessoaNome", "")

            # 2. Buscar ficha completa para ter acesso ao historicoDelito
            url_ficha = "http://172.18.4.46:8080/sispen_api/custodiado/ficha"
            payload_ficha = {
                "idPessoa": str(id_real),
                "idUsuario": 3104,
                "idUnidade": 157,
                "idOrigem": 1,
                "unidadesAcesso": [170,157]
            }
            resp_ficha = requests.post(url_ficha, headers=headers, json=payload_ficha)
            if resp_ficha.status_code == 200:
                ficha = resp_ficha.json()
                custodiado = ficha.get("custodiado")

            # 3. Buscar processos pelo endpoint específico
            url_processos = "http://172.18.4.46:8080/sispen_api/custodiado/ficha/processos"
            payload_processos = {
                "idPessoa": str(id_real),
                "idUsuario": 3104,
                "idUnidade": 157,
                "idOrigem": 1,
                "unidadesAcesso": [170,157]
            }
            resp_processos = requests.post(url_processos, headers=headers, json=payload_processos)

            if resp_processos.status_code == 200:
                processos = resp_processos.json().get("processos", [])
                recolhidos = [
                    p for p in processos
                    if p.get("tipoSituacaoPrisional", {}).get("descricao", "").strip().lower() == "recolhido por este processo".lower()
                ]
                if recolhidos:
                    numero_processo = recolhidos[0].get("numeroProcesso", "")
                elif processos:
                    numero_processo = processos[0].get("numeroProcesso", "")

            # 4. Se ainda ficou "Sem processo", tenta extrair do historicoDelito
            if numero_processo == "Sem processo" and custodiado:
                historico = custodiado.get("historicoDelito", "")
                if historico:
                    match = re.findall(r"\d{7}-\d{2}\.\d{4}\.\d\.\d{2}\.\d{4}", historico)
                    if match:
                        numero_processo = match[0]

        else:
            print(f"{pessoa_id} - Erro custodiado {resp_real.status_code}")
        
        # monta linha para planilha
        resultados.append({ 
            "Prontuário": pessoa_id, 
            "Nome": pessoa_nome, 
            "Processo Recolhido": numero_processo 
        })

        print(f"{pessoa_id} - Sucesso na requisição {numero_processo}")

    except Exception as e:
        print(f"{pessoa_id} - Erro na requisição: {e}")

    time.sleep(1)  # se quiser diminuir velocidade, descomente

# cria DataFrame final 
df_final = pd.DataFrame(resultados) 

# salva em Excel 
saida = r"D:\Cotic\resultado_UPF.xlsx" 
df_final.to_excel(saida, index=False) 
print(f"Planilha gerada em: {saida}")


Total de IDs capturados: 182
219429 - Sucesso na requisição Sem processo
102724 - Sucesso na requisição Sem processo
525121 - Sucesso na requisição Sem processo
550824 - Sucesso na requisição Sem processo
146217 - Sucesso na requisição Sem processo
563207 - Sucesso na requisição Sem processo
426840 - Sucesso na requisição Sem processo
586236 - Sucesso na requisição Sem processo
500592 - Sucesso na requisição Sem processo
591044 - Sucesso na requisição Sem processo
512311 - Sucesso na requisição Sem processo
510808 - Sucesso na requisição Sem processo
397122 - Sucesso na requisição Sem processo
542847 - Sucesso na requisição Sem processo
582822 - Sucesso na requisição Sem processo
554278 - Sucesso na requisição Sem processo
552482 - Sucesso na requisição Sem processo
541239 - Sucesso na requisição Sem processo
525108 - Sucesso na requisição Sem processo
533599 - Sucesso na requisição Sem processo
550230 - Sucesso na requisição Sem processo
581521 - Sucesso na requisição Sem processo
375

KeyboardInterrupt: 